In [ ]:
import pdfplumber
from PIL import Image
from datasets import Dataset
import torch
from transformers import (
    AutoProcessor,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
)

In [ ]:
pdf_path = "./files/output2.pdf"
dataset_list = []

words = []
bboxes = []
with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages):
        # Convert page to PIL image
        img = page.to_image(resolution=350).original

        # Extract words + bbox
        for word in page.extract_words():
            words.append(word["text"])
            # Normalize bbox to 0-1000
            x0 = int(word["x0"] / page.width * 1000)
            y0 = int(word["top"] / page.height * 1000)
            x1 = int(word["x1"] / page.width * 1000)
            y1 = int(word["bottom"] / page.height * 1000)
            bboxes.append([x0, y0, x1, y1])

        # Labels (0=O for unsupervised; replace with manual annotations for NER)
        labels = [0] * len(words)

        dataset_list.append(
            {"image": img, "words": words, "bboxes": bboxes, "labels": labels}
        )

dataset = Dataset.from_list(dataset_list)

In [ ]:
import fitz  # PyMuPDF

pdf_path = "./files/output2.pdf"
output_pdf_path = "./files/with_boxes.pdf"

# Open the original PDF
doc = fitz.open(pdf_path)

for page_idx, data in enumerate(dataset_list):
    page = doc[page_idx]
    words = data["words"]
    bboxes = data["bboxes"]

    for bbox in bboxes:
        # Denormalize bbox back to PDF coordinates
        x0 = bbox[0] / 1000 * page.rect.width
        y0 = bbox[1] / 1000 * page.rect.height
        x1 = bbox[2] / 1000 * page.rect.width
        y1 = bbox[3] / 1000 * page.rect.height

        # Draw rectangle
        rect = fitz.Rect(x0, y0, x1, y1)
        page.draw_rect(rect, color=(1, 0, 0), width=1)  # red box

# Save the PDF
doc.save(output_pdf_path)
doc.close()

print(f"Saved output PDF with boxes: {output_pdf_path}")

In [ ]:
id = 1
labels = []
for word, bbox in zip(words, bboxes):
    print(f"id: [{id}] -> {word} | BBox: {bbox}")
    labels.append(word)
    id += 1

In [ ]:
model_name = "microsoft/layoutlmv3-base"

# ⚠ apply_ocr=False here
processor = AutoProcessor.from_pretrained(model_name, apply_ocr=False)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=10,  # adjust for your task
)

In [ ]:
def preprocess(batch):
    encoded = processor(
        images=batch["image"],
        text=batch["words"],
        boxes=batch["bboxes"],
        word_labels=batch["labels"],
        return_tensors="pt",
        padding="max_length",
        truncation=True,
    )
    return encoded


encoded_dataset = dataset.map(
    preprocess, batched=True, remove_columns=dataset.column_names
)

In [ ]:
encoded_dataset

In [ ]:
outputDIR = "./layoutlmv3-finetuned_III"

In [ ]:
args = TrainingArguments(
    output_dir=outputDIR,
    per_device_train_batch_size=1,  # for single PDF
    learning_rate=5e-5,
    weight_decay=0.01,
    num_train_epochs=5,
    logging_steps=1,
    save_steps=10,
    # fp16=torch.backends.mps.is_available(),
    report_to="none",
    use_mps_device=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded_dataset,
)

In [ ]:
trainer.train()
model.save_pretrained(outputDIR)
processor.save_pretrained(outputDIR)

In [ ]:
model_dir = "./layoutlmv3-finetuned"
processor = AutoProcessor.from_pretrained(model_dir, apply_ocr=False)
model = AutoModelForTokenClassification.from_pretrained(model_dir)

In [ ]:
model.eval()  # set to evaluation mode
pdf_path = "./files/output.pdf"
# pdf_path = "./files/epfo_passbook.pdf"
# pdf_path = "./files/a_01.pdf

In [ ]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]  # first page
    img = page.to_image(resolution=300).original

    words = []
    bboxes = []
    for word in page.extract_words():
        words.append(word["text"])
        x0 = int(word["x0"] / page.width * 1000)
        y0 = int(word["top"] / page.height * 1000)
        x1 = int(word["x1"] / page.width * 1000)
        y1 = int(word["bottom"] / page.height * 1000)
        bboxes.append([x0, y0, x1, y1])

In [ ]:
inputs = processor(
    images=img,
    text=words,
    boxes=bboxes,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
)

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

In [ ]:
predictions = torch.argmax(outputs.logits, dim=-1).squeeze().tolist()

In [ ]:
id = 1
labels = []
for word, label_id, bbox in zip(words, predictions, bboxes):
    print(f"id: [{id}] -> {word} | Label: {label_id} | BBox: {bbox}")
    labels.append(word)
    id += 1

In [ ]:
labels

In [ ]:
labels[98]

In [ ]:
import torch
import pdfplumber
from transformers import AutoProcessor, AutoModelForTokenClassification
from io import BytesIO
import json

# 1. Load model and processor
model_name = "microsoft/layoutlmv3-base"
processor = AutoProcessor.from_pretrained(model_name, apply_ocr=False)

id2label = {
    0: "O",
    1: "B-CUST_NAME",
    2: "I-CUST_NAME",
    3: "B-ADDRESS",
    4: "I-ADDRESS",
    5: "B-CONTACT",
    6: "I-CONTACT",
    7: "B-INVOICE_NO",
    8: "I-INVOICE_NO",
    9: "B-PAYMENT_REF",
    10: "I-PAYMENT_REF",
    11: "B-ORDER_REF",
    12: "I-ORDER_REF",
    13: "B-DATE",
    14: "I-DATE",
    15: "B-MODE_PAYMENT",
    16: "I-MODE_PAYMENT",
    17: "B-ITEM",
    18: "I-ITEM",
    19: "B-QTY",
    20: "I-QTY",
    21: "B-AMOUNT",
    22: "I-AMOUNT",
    23: "B-TAXABLE_AMT",
    24: "I-TAXABLE_AMT",
    25: "B-TAX",
    26: "I-TAX",
    27: "B-TOTAL",
    28: "I-TOTAL",
    29: "B-OFFICE_NAME",
    30: "I-OFFICE_NAME",
    31: "B-OFFICE_ADDR",
    32: "I-OFFICE_ADDR",
    33: "B-CIN",
    34: "I-CIN",
    35: "B-PAN",
    36: "I-PAN",
    37: "B-GST",
    38: "I-GST",
    39: "B-DECLARATION",
    40: "I-DECLARATION",
    41: "B-DIGITAL_SIGN",
    42: "I-DIGITAL_SIGN",
    43: "B-WEBSITE",
    44: "I-WEBSITE",
}
# label2id = {v: k for k, v in id2label.items()}

label2id = {v: k for k, v in id2label.items()}

model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=len(id2label), id2label=id2label, label2id=label2id
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 2. Read PDF
pdf_path = "./files/_000001.pdf"
words, boxes = [], []
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    width, height = page.width, page.height
    for w in page.extract_words():
        words.append(w["text"])
        x0 = int(w["x0"] / width * 1000)
        y0 = int(w["top"] / height * 1000)
        x1 = int(w["x1"] / width * 1000)
        y1 = int(w["bottom"] / height * 1000)
        boxes.append([x0, y0, x1, y1])
    image = page.to_image(resolution=300).original

# 3. Encode
encoding = processor(
    images=image,
    text=words,
    boxes=boxes,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
)
encoding = {k: v.to(device) for k, v in encoding.items()}

# 4. Run inference
with torch.no_grad():
    outputs = model(**encoding)

preds = outputs.logits.argmax(-1).squeeze().tolist()

# 5. Group tokens into entities
entities = []
current = {"label": None, "text": "", "bbox": None}

for word, pred_id, bbox in zip(words, preds, boxes):
    label = id2label[pred_id]

    if label.startswith("B-"):
        if current["text"]:
            entities.append(current)
        current = {"label": label[2:], "text": word, "bbox": bbox}

    elif label.startswith("I-") and current["text"]:
        current["text"] += " " + word
        x0 = min(current["bbox"][0], bbox[0])
        y0 = min(current["bbox"][1], bbox[1])
        x1 = max(current["bbox"][2], bbox[2])
        y1 = max(current["bbox"][3], bbox[3])
        current["bbox"] = [x0, y0, x1, y1]

    else:
        if current["text"]:
            entities.append(current)
            current = {"label": None, "text": "", "bbox": None}

if current["text"]:
    entities.append(current)

# 6. Output JSON
structured_data = {}
for ent in entities:
    structured_data.setdefault(ent["label"], []).append(
        {"text": ent["text"], "bbox": ent["bbox"]}
    )

print(json.dumps(structured_data, indent=2))

In [ ]:
labeled_words = []

for word, pred_id in zip(words, preds):
    label = id2label[pred_id]
    labeled_words.append((word, label))

print(labeled_words)